In [6]:
import os
from dotenv import load_dotenv

print("Thư mục hiện tại:", os.getcwd())

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    print("Đã chuyển thư mục ra root:", os.getcwd())

load_dotenv()
print("Đã load file .env thành công!")

Thư mục hiện tại: /home/huyy-giaa/Develop/VietCulture_Rag
Đã load file .env thành công!


In [11]:
import json
import pandas as pd
from tqdm import tqdm
import time
from src.agent.graph import create_agent_bundle, invoke_agent 
from src.retrieval import qa_retriever
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

In [12]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import os

# --- 1. SETUP GIÁM KHẢO GEMINI ---
# Hãy đảm bảo bạn đã load biến môi trường GOOGLE_API_KEY
# Dùng gemini-1.5-flash làm giám khảo cho tốc độ nhanh và chi phí rẻ
eval_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
eval_embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base"
)

# --- 2. LOAD DỮ LIỆU ĐÃ CHẠY (TEST 3 CÂU) ---
# Bạn thay tên file thành test10.json nếu muốn test 10 câu
df_results = pd.read_json("notebooks/eval_results_test3.json")

# RAGAS yêu cầu format data chuẩn bị đưa vào. 
# Cấu trúc JSON của bạn đã khớp tên cột rồi, chỉ cần chuyển qua HuggingFace Dataset
eval_dataset = Dataset.from_pandas(df_results[["question", "answer", "contexts", "ground_truth"]])

print("Đã nạp xong Dataset cho RAGAS!")
print(eval_dataset)

# --- 3. ĐƯA LÊN BÀN CHẤM ---
print("\nGiám khảo RAGAS đang chấm điểm... (sẽ mất khoảng 1-2 phút cho 3 câu)")

metrics = [
    faithfulness,       # Bot trả lời có bịa đặt không?
    answer_relevancy,   # Trả lời có đúng trọng tâm câu hỏi không?
    context_precision,  # Tài liệu mang về có sạch/chuẩn không?
    context_recall      # Tài liệu mang về có đủ ý trả lời không?
]

# Hàm evaluate của RAGAS
score = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=eval_llm,
    embeddings=eval_embeddings
)

# --- 4. XUẤT KẾT QUẢ ---
print("\n=== ĐIỂM SỐ TỔNG QUAN ===")
print(score)

# Chuyển kết quả chi tiết thành DataFrame để dễ xem
df_ragas_details = score.to_pandas()

# Bạn có thể xuất ra file CSV hoặc JSON để lưu lại
df_ragas_details.to_csv("notebooks/ragas_test_score.csv", index=False)
print("\nĐã lưu kết quả chi tiết ra file: notebooks/ragas_test_score.csv")

/tmp/ipykernel_37838/3948104344.py:4: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_37838/3948104344.py:4: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipykernel_37838/3948104344.py:4: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
/tmp/ipykernel_37838/3948104344.py:4: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be rem

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Đã nạp xong Dataset cho RAGAS!
Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 9
})

Giám khảo RAGAS đang chấm điểm... (sẽ mất khoảng 1-2 phút cho 3 câu)


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

The LLM did not return a valid classification.
The LLM did not return a valid classification.
The LLM did not return a valid classification.
The LLM did not return a valid classification.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
The LLM did not return a valid classification.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
The LLM did not return a valid classification.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
The LLM did not return a valid classification.
The LLM did not return a valid classification.
LLM returned 1 generations instead of requested 3.


=== ĐIỂM SỐ TỔNG QUAN ===
{'faithfulness': 0.9365, 'answer_relevancy': 0.9235, 'context_precision': 0.0000, 'context_recall': nan}

Đã lưu kết quả chi tiết ra file: notebooks/ragas_test_score.csv


In [10]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

/home/huyy-giaa/miniconda3/envs/ds102-th/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
E0000 00:00:1782497291.818871   37838 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1782497291.818890   37838 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1782497291.818891   37838 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1782497291.818892 

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr